# D2.5 · Replay and forensics

**Function D — Security Operations → The Incident Responder**  ·  *Security of AI*

---

**Risk.** Non-determinism as an evidentiary problem.

**Control.** Log at design time what replay will need.

**This lab.** Replay an agent run to a regulator-grade standard.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D2.5"))

Replay and forensics. Deterministic replay is the difference between demonstrating what happened and describing it.

In [ ]:
from cybercommons import ir

CONFIGS = {
 "fully instrumented": ir.Replay(prompts=["fix the config"],
                                 tool_results=["file contents…"],
                                 model_version="glm-4.6@2025-11", seed=42),
 "prompts only":       ir.Replay(prompts=["fix the config"]),
 "typical production": ir.Replay(prompts=["fix the config"],
                                 tool_results=["file contents…"]),
 "nothing":            ir.Replay(),
}
for name, r in CONFIGS.items():
    ok, missing = r.replayable()
    print(f"{name:22s} replayable={str(ok):5s}")
    for m in missing:
        print(f"{'':24s}✗ {m}")

The `typical production` row is the one to sit with: prompts and tool results are recorded, and the run is still not replayable because the model version was never pinned. A silent provider-side upgrade invalidates every reconstruction made after it.

### Expect

Only the fully instrumented configuration is replayable. The typical production row is missing the pinned model version and the seed.

### Your turn

Which of the four fields is cheapest to start recording today? Model version is usually one line and it is the one that silently invalidates the others.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D2.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*